In [1]:
%pip install eyepop==3.12.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.6/89.6 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 6.2 MB/s eta 0:00:00
  Attempting uninstall: cryptography
    Found existing installation: cryptography 49.0.0
    Uninstalling cryptography-49.0.0:
      Successfully uninstalled cryptography-49.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pyopenssl 26.3.0 requires cryptography<50,>=49.0.0, but you have cryptography 46.0.7 which is incompatible.


In [2]:
import getpass

EYEPOP_ACCOUNT_ID=input("Enter your Account UUID: ")
EYEPOP_API_KEY=getpass.getpass('Enter your API KEY: ')


Enter your Account UUID: a5184defa8e847248f589d35080efbfa
Enter your API KEY: ··········


In [3]:
NAMESPACE_PREFIX="datasciencealliance-org" # Add your namespace-prefix here

### Define Ability

In [4]:
from eyepop import EyePopSdk
from eyepop.data.data_types import InferRuntimeConfig, VlmAbilityGroupCreate, VlmAbilityCreate, TransformInto
from eyepop.worker.worker_types import InferenceComponent, Pop
import json


QSR_CUSTOMER_EXPERIENCE_PROMPT = """
Determine the primary customer experience issue shown in the quick service restaurant image and assign exactly one label from this list:

['Long_Line', 'Guests_Unable_To_Find_Seating', 'Dirty_Tables_Affecting_Seating_Availability', 'Lobby_Crowded', 'Beverage_Station_Congestion']

Classify based on the single most dominant visible customer experience issue.

Choose 'Long_Line' only when the main issue is a clear queue of guests waiting at the ordering counter, cashier, kiosk, pickup counter, or service area. The people should form a visible line. Do not choose this label just because many people are standing around.

Choose 'Guests_Unable_To_Find_Seating' when the main issue is guests holding food, trays, bags, or drinks while looking for a place to sit because most tables or seats are occupied. This label should be chosen when guests are standing near the dining area with food and appear unable to find open seating. If several guests are holding trays or food and looking around the dining area, choose this label even if the restaurant also looks busy. Do not choose 'Lobby_Crowded' just because many tables are occupied.

Choose 'Dirty_Tables_Affecting_Seating_Availability' when the main issue is that tables are empty but unusable because they have leftover trays, wrappers, cups, napkins, crumbs, spilled food, or trash on them. If guests are looking for seats and the visible open tables are dirty, choose this label instead of 'Guests_Unable_To_Find_Seating'.

Choose 'Lobby_Crowded' only when the main issue is general congestion across the lobby, entrance, waiting area, pickup area, or walkways. Use this label when many guests are standing or blocking movement across the whole lobby, but there is no clear long line, no clear seating search behavior, no dirty-table seating issue, and no beverage station bottleneck.

Choose 'Beverage_Station_Congestion' when the main issue is crowding around the self-serve beverage station, soda fountain, drink dispenser, ice machine, cup lids, straws, napkins, or condiment area. Use this label when the congestion is clearly localized around the drink station.

Decision priority:
1. If the main crowd is around the beverage station, choose 'Beverage_Station_Congestion'.
2. If open tables are dirty and unusable, choose 'Dirty_Tables_Affecting_Seating_Availability'.
3. If guests are holding trays or food and searching for seats, choose 'Guests_Unable_To_Find_Seating'.
4. If people form a clear line, choose 'Long_Line'.
5. If people are generally spread across the lobby and blocking movement, choose 'Lobby_Crowded'.

Important distinction:
'Guests_Unable_To_Find_Seating' is about people with food looking for seats.
'Lobby_Crowded' is about general standing congestion in the lobby or walkway.
Occupied tables alone do not mean 'Lobby_Crowded'. If the main visible issue is that guests with trays cannot find seats, choose 'Guests_Unable_To_Find_Seating'.

Ignore brand logos, menu text, signs, uniforms, readable text, timestamps, camera overlays, and decorative elements. Classify based only on the visible restaurant condition.

Return only the single best-fitting label.
"""


ability_prototypes = [
    VlmAbilityCreate(
        name=f"{NAMESPACE_PREFIX}.image-classify.qsr-customer-experience-monitor",
        description="Classify quick service restaurant lobby images based on visible customer experience issues such as long lines, seating problems, dirty tables, crowding, or beverage station congestion",
        worker_release="qwen3-instruct",
        text_prompt=QSR_CUSTOMER_EXPERIENCE_PROMPT,
        transform_into=TransformInto(),
        config=InferRuntimeConfig(
            max_new_tokens=20,
            image_size=640
        ),
        is_public=False
    )
]

### Create Ability

In [5]:
with EyePopSdk.dataEndpoint(api_key=EYEPOP_API_KEY, account_id=EYEPOP_ACCOUNT_ID) as endpoint:
    for ability_prototype in ability_prototypes:
        ability_group = endpoint.create_vlm_ability_group(VlmAbilityGroupCreate(
            name=ability_prototype.name,
            description=ability_prototype.description,
            default_alias_name=ability_prototype.name,
        ))
        ability = endpoint.create_vlm_ability(
            create=ability_prototype,
            vlm_ability_group_uuid=ability_group.uuid,
        )
        ability = endpoint.publish_vlm_ability(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
        )
        ability = endpoint.add_vlm_ability_alias(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
            tag_name="latest"
        )
        print(f"created ability {ability.uuid} with alias entries {ability.alias_entries}")

created ability 06a5e933a1767b3780005bf2e2a68f42 with alias entries [AbilityAliasEntry(alias='datasciencealliance-org.image-classify.qsr-customer-experience-monitor', tag='1.0.2'), AbilityAliasEntry(alias='datasciencealliance-org.image-classify.qsr-customer-experience-monitor', tag='latest')]


### Evaluate with Single QSR Image

In [7]:
from pathlib import Path
import json
from eyepop import EyePopSdk
from eyepop.worker.worker_types import InferenceComponent, Pop


QSR_LABELS = [
    "Long_Line",
    "Guests_Unable_To_Find_Seating",
    "Dirty_Tables_Affecting_Seating_Availability",
    "Lobby_Crowded",
    "Beverage_Station_Congestion"
]


def extract_top_class(result):
    classes = result.get("classes", [])

    if classes:
        top_class = max(classes, key=lambda item: item.get("confidence", 0))
        return top_class.get("classLabel"), top_class.get("confidence")

    # Fallback in case the result returns text instead of classes.
    for text_item in result.get("texts", []):
        text = text_item.get("text", "").strip().strip('"').strip("'")
        if text in QSR_LABELS:
            return text, None

    return None, None


pop = Pop(components=[
    InferenceComponent(
        ability=f"{NAMESPACE_PREFIX}.image-classify.qsr-customer-experience-monitor:latest"
    )
])


# Replace this with your actual image path.
input_path = Path("/content/sample_qsr_image.jpg")


raw_results = []

with EyePopSdk.workerEndpoint(api_key=EYEPOP_API_KEY) as endpoint:
    endpoint.set_pop(pop)

    job = endpoint.upload(input_path)

    while result := job.predict():
        raw_results.append(result)


print("=== RAW EYEPOP RESULT ===")
print(json.dumps(raw_results[:1], indent=2))


print("\n=== QSR CUSTOMER EXPERIENCE PREDICTION ===")

if raw_results:
    label, confidence = extract_top_class(raw_results[0])

    if label:
        print(f"Predicted label: {label}")

        if confidence is not None:
            print(f"Confidence: {confidence:.4f}")
    else:
        print("No valid QSR label detected.")
else:
    print("No result returned.")

=== RAW EYEPOP RESULT ===
[
  {
    "compute_units": 0.1,
    "seconds": 0,
    "source_height": 559,
    "source_id": "5cc679c1-8482-11f1-ba8b-96e6a9782450",
    "source_width": 1024,
    "system_timestamp": 1784583090473159000,
    "texts": [
      {
        "id": 1,
        "text": "Lobby_Crowded"
      }
    ],
    "timestamp": 0
  }
]

=== QSR CUSTOMER EXPERIENCE PREDICTION ===
Predicted label: Lobby_Crowded
